# SRCNet metadata: 100-source TAP capability demo

This notebook runs against the Docker Compose deployment and showcases the public API end to end: OpenAPI/VOSI/TAP_SCHEMA discovery, model validation, bulk ingestion, JSON and PyVO queries, spatial ADQL, joins, synchronous and asynchronous execution, JSON job lifecycle, list/fetch, amendment, and cascading deletion. Stable identifiers make ingestion safe to rerun.

In [1]:
import copy
import os
import time

import pyvo
import requests
from ska_src_mm_notification.builder import NotificationBuilder
from ska_src_mm_notification.models.schemas.srcnet_ingestion import Artifact, DataProduct
from ska_src_sdm import Software

TAP_URL = os.environ.get("TAP_URL", "http://localhost:8080/tap").rstrip("/")
API_URL = TAP_URL.removesuffix("/tap") + "/api/v1"
PROJECT_ID = "demo-100-sources"
SOFTWARE_URI = "ska:srcnet-demo:1.0.0"

availability = requests.get(f"{TAP_URL}/availability", timeout=10)
availability.raise_for_status()
tap = pyvo.dal.TAPService(TAP_URL)
print(f"Service available at {TAP_URL}")

Service available at http://localhost:8080/tap


## 1. Discover the service and model-generated tables

PyVO reads the VOSI `/tables` document, which is backed by `TAP_SCHEMA`.

In [2]:
openapi = requests.get(f"{API_URL.removesuffix('/api/v1')}/openapi.json", timeout=10).json()
json_tables = requests.get(f"{API_URL}/tables", timeout=20).json()["tables"]
print(
    f"OpenAPI advertises {len(openapi['paths'])} paths; "
    f"JSON metadata lists {len(json_tables)} tables"
)

for table_name in ("srcnet.data_products", "srcnet.artifacts", "srcnet.software"):
    table = tap.tables[table_name]
    print(f"{table_name}: {len(table.columns)} discoverable columns")

srcnet_tables = tap.search(
    "SELECT table_name, description FROM TAP_SCHEMA.tables "
    "WHERE schema_name = 'srcnet' ORDER BY table_name"
).to_table()
srcnet_tables

OpenAPI advertises 27 paths; JSON metadata lists 16 tables
srcnet.data_products: 39 discoverable columns
srcnet.artifacts: 23 discoverable columns
srcnet.software: 17 discoverable columns


table_name,description
object,object
srcnet.artifacts,"Represents a single data artifact (file) within a data product. Artifacts are\nthe actual files that will be ingested into the system. Spatial, spectral, and\ntemporal coverage fields are specific to this artifact."
srcnet.data_products,"Represents a data product containing one or more artifacts. Data products group\nrelated artifacts together with common metadata. Spatial, spectral, and temporal\ncoverage at this level are computed from the artifacts (centroids for spatial,\nunion for spectral/temporal)."
srcnet.execution_blocks,Represents an execution block containing data products. An execution block is a\nsingle execution of a scheduling block.
srcnet.observations,"Represents an observation containing scheduling blocks. An observation groups\nrelated scheduling blocks with common instrument metadata. Spatial, spectral,\nand temporal coverage are defined at the DataProduct and Artifact levels."
srcnet.projects,Root model for SRC Ingestion Notification files. This represents the\nhierarchical structure: project_id -> obs_id -> sbd_id -> eb_id -> product_id ->\nartifact_id The unique identifier for data products is constructed from:\neb_id.product_id
srcnet.scheduling_blocks,Represents a scheduling block containing execution blocks. A scheduling block\ndefines a set of observations to be executed.
srcnet.software,software
srcnet.software_artifacts,software_artifacts


## 2. Build and ingest 100 sources

Each source is represented by a data product with a distinct sky position and one artifact. The notification library validates all 100 records before they are posted.

In [3]:
centre_ra, centre_dec = 62.3, -65.5
products = []
for index in range(100):
    row, column = divmod(index, 10)
    ra = centre_ra + (column - 4.5) * 0.12
    dec = centre_dec + (row - 4.5) * 0.10
    source_id = f"demo-source-{index:03d}"
    products.append(
        DataProduct(
            product_id=source_id,
            o_ucd="phot.flux.density",
            dataproduct_type="image",
            calib_level=2,
            target_name=f"SRCNet source {index:03d}",
            s_ra=ra,
            s_dec=dec,
            s_fov=0.05,
            artifacts=[
                Artifact(
                    artifact_id=f"{source_id}-fits",
                    access_url=f"https://example.org/{source_id}.fits",
                    access_format="application/fits",
                    access_estsize=1024 * (index + 1),
                    s_ra=ra,
                    s_dec=dec,
                    s_fov=0.05,
                )
            ],
        )
    )

notification = (
    NotificationBuilder()
    .create_simple_notification(
        project_id=PROJECT_ID,
        group_ids=["demo-users"],
        obs_id="demo-observation",
        obs_title="100-source TAP demonstration",
        sbd_id="demo-scheduling-block",
        eb_id="demo-execution-block",
        data_products=products,
        project_title="SRCNet 100-source demonstration",
        pi_name="Demo User",
        instrument_name="SKA",
        facility_name="SKAO",
    )
    .build_dict()
)
response = requests.post(f"{API_URL}/notifications", json=notification, timeout=60)
response.raise_for_status()
ingest_result = response.json()
assert ingest_result["rows"]["srcnet.data_products"] == 100
assert ingest_result["rows"]["srcnet.artifacts"] == 100
ingest_result

{'status': 'ingested',
 'project_id': 'demo-100-sources',
 'rows': {'srcnet.projects': 1,
  'srcnet.observations': 1,
  'srcnet.scheduling_blocks': 1,
  'srcnet.execution_blocks': 1,
  'srcnet.data_products': 100,
  'srcnet.artifacts': 100},
 'query_hint': "SELECT * FROM srcnet.projects WHERE project_id = 'demo-100-sources' (via /tap/sync or /api/v1/query)"}

### Validation errors are structured and non-destructive

In [4]:
invalid_notification = copy.deepcopy(notification)
invalid_notification["observations"][0]["scheduling_blocks"][0]["execution_blocks"][0][
    "data_products"
][0]["s_dec"] = 95.0
rejected = requests.post(f"{API_URL}/notifications", json=invalid_notification, timeout=30)
assert rejected.status_code == 422
rejected.json()["detail"][0]

{'type': 'less_than_equal',
 'loc': ['body',
  'observations',
  0,
  'scheduling_blocks',
  0,
  'execution_blocks',
  0,
  'data_products',
  0,
  's_dec'],
 'msg': 'Input should be less than or equal to 90',
 'input': 95.0,
 'ctx': {'le': 90.0}}

## 3. Discover ingested documents through the JSON API

In [5]:
projects = requests.get(f"{API_URL}/notifications", timeout=20).json()["projects"]
project_summary = next(project for project in projects if project["project_id"] == PROJECT_ID)
assert project_summary["data_products"] == 100
assert project_summary["artifacts"] == 100
document = requests.get(f"{API_URL}/notifications/{PROJECT_ID}", timeout=30).json()
stored_products = document["observations"][0]["scheduling_blocks"][0]["execution_blocks"][0][
    "data_products"
]
assert len(stored_products) == 100
project_summary

{'pi_name': 'Demo User',
 'group_ids': ['demo-users'],
 'project_id': 'demo-100-sources',
 'project_title': 'SRCNet 100-source demonstration',
 'schema_version': '2.0',
 'observations': 1,
 'scheduling_blocks': 1,
 'execution_blocks': 1,
 'data_products': 100,
 'artifacts': 100}

## 4. Query through the synchronous JSON API

In [6]:
json_response = requests.post(
    f"{API_URL}/query",
    json={
        "query": (
            "SELECT TOP 5 product_id, s_ra, s_dec FROM srcnet.data_products "
            f"WHERE project_id = '{PROJECT_ID}' ORDER BY product_id"
        ),
        "maxrec": 5,
    },
    timeout=30,
)
json_response.raise_for_status()
json_result = json_response.json()
assert json_result["status"] == "OK" and len(json_result["data"]) == 5
json_result

{'metadata': [{'name': 'product_id',
   'datatype': 'char',
   'unit': None,
   'ucd': None,
   'description': 'Unique identifier for the data product'},
  {'name': 's_ra',
   'datatype': 'double',
   'unit': None,
   'ucd': None,
   'description': 'Right ascension of product center in degrees (computed from artifacts)'},
  {'name': 's_dec',
   'datatype': 'double',
   'unit': None,
   'ucd': None,
   'description': 'Declination of product center in degrees (computed from artifacts)'}],
 'data': [['demo-source-000', 61.76, -65.95],
  ['demo-source-001', 61.879999999999995, -65.95],
  ['demo-source-002', 62.0, -65.95],
  ['demo-source-003', 62.12, -65.95],
  ['demo-source-004', 62.239999999999995, -65.95]],
 'status': 'OK'}

## 5. Query and spatially discover sources with PyVO

In [7]:
all_sources = tap.search(
    "SELECT product_id, target_name, s_ra, s_dec, calib_level "
    "FROM srcnet.data_products "
    f"WHERE project_id = '{PROJECT_ID}' ORDER BY product_id"
).to_table()
assert len(all_sources) == 100
all_sources[:10]

product_id,target_name,s_ra,s_dec,calib_level
object,object,float64,float64,int64
demo-source-000,SRCNet source 000,61.76,-65.95,2
demo-source-001,SRCNet source 001,61.879999999999995,-65.95,2
demo-source-002,SRCNet source 002,62.0,-65.95,2
demo-source-003,SRCNet source 003,62.12,-65.95,2
demo-source-004,SRCNet source 004,62.239999999999995,-65.95,2
demo-source-005,SRCNet source 005,62.36,-65.95,2
demo-source-006,SRCNet source 006,62.48,-65.95,2
demo-source-007,SRCNet source 007,62.599999999999994,-65.95,2
demo-source-008,SRCNet source 008,62.72,-65.95,2


In [8]:
cone_results = tap.search(
    "SELECT product_id, target_name, s_ra, s_dec "
    "FROM srcnet.data_products "
    f"WHERE project_id = '{PROJECT_ID}' AND "
    f"1 = CONTAINS(POINT('ICRS', s_ra, s_dec), CIRCLE('ICRS', {centre_ra}, {centre_dec}, 0.3)) "
    "ORDER BY product_id"
).to_table()
print(f"Cone search returned {len(cone_results)} of 100 sources")
cone_results

Cone search returned 52 of 100 sources


product_id,target_name,s_ra,s_dec
object,object,float64,float64
demo-source-022,SRCNet source 022,62.0,-65.75
demo-source-023,SRCNet source 023,62.12,-65.75
demo-source-024,SRCNet source 024,62.239999999999995,-65.75
demo-source-025,SRCNet source 025,62.36,-65.75
demo-source-026,SRCNet source 026,62.48,-65.75
demo-source-027,SRCNet source 027,62.599999999999994,-65.75
demo-source-030,SRCNet source 030,61.76,-65.65
demo-source-031,SRCNet source 031,61.879999999999995,-65.65
demo-source-032,SRCNet source 032,62.0,-65.65


## 6. Run an asynchronous TAP query and join artifacts

In [9]:
async_result = tap.run_async(
    "SELECT TOP 20 p.product_id, p.s_ra, p.s_dec, a.access_url, a.access_estsize "
    "FROM srcnet.data_products AS p "
    "JOIN srcnet.artifacts AS a ON a.project_id = p.project_id "
    "AND a.obs_id = p.obs_id AND a.sbd_id = p.sbd_id "
    "AND a.eb_id = p.eb_id AND a.product_id = p.product_id "
    f"WHERE p.project_id = '{PROJECT_ID}' ORDER BY p.product_id"
).to_table()
assert len(async_result) == 20
async_result

product_id,s_ra,s_dec,access_url,access_estsize
object,float64,float64,object,int64
demo-source-000,61.76,-65.95,https://example.org/demo-source-000.fits,1024
demo-source-001,61.879999999999995,-65.95,https://example.org/demo-source-001.fits,2048
demo-source-002,62.0,-65.95,https://example.org/demo-source-002.fits,3072
demo-source-003,62.12,-65.95,https://example.org/demo-source-003.fits,4096
demo-source-004,62.239999999999995,-65.95,https://example.org/demo-source-004.fits,5120
demo-source-005,62.36,-65.95,https://example.org/demo-source-005.fits,6144
demo-source-006,62.48,-65.95,https://example.org/demo-source-006.fits,7168
demo-source-007,62.599999999999994,-65.95,https://example.org/demo-source-007.fits,8192
demo-source-008,62.72,-65.95,https://example.org/demo-source-008.fits,9216


## 7. Ingest and discover software metadata

In [10]:
software_payload = {
    "uri": SOFTWARE_URI,
    "description": "Software metadata inserted by the TAP demo notebook",
    "release_date": "2026-01-15T00:00:00Z",
    "status": "STABLE",
    "artifacts": [
        {
            "kind": "DOCKER",
            "location": "ghcr.io/ska-telescope/srcnet-demo:1.0.0",
            "cpu_architecture": ["amd64"],
            "supported_modes": ["HEADLESS", "NOTEBOOK"],
        }
    ],
    "discovery": {"science_category": ["Demo"], "tools_included": ["pyvo"]},
    "resources": {"requires_gpu": False, "min_memory": 2},
    "provenance": {"registered_by": "demo-notebook"},
}
software = Software.from_dict(software_payload)
response = requests.post(f"{API_URL}/software", json=software.to_dict(), timeout=30)
response.raise_for_status()
software_rows = tap.search(
    "SELECT s.uri, s.description, s.status, a.kind, a.location "
    "FROM srcnet.software AS s JOIN srcnet.software_artifacts AS a ON a.uri = s.uri "
    f"WHERE s.uri = '{SOFTWARE_URI}'"
).to_table()
software_rows

uri,description,status,kind,location
object,object,object,object,object
ska:srcnet-demo:1.0.0,Software metadata inserted by the TAP demo notebook,STABLE,DOCKER,ghcr.io/ska-telescope/srcnet-demo:1.0.0


## 8. Amend one source

The PATCH endpoint validates amended values against the same data model and scopes the update to this project.

In [11]:
amended_source_id = "demo-source-000"
amendment = requests.patch(
    f"{API_URL}/notifications/{PROJECT_ID}",
    json={
        "table": "data_products",
        "match": {"product_id": amended_source_id},
        "values": {"target_name": "Amended SRCNet source", "s_ra": 62.31, "s_dec": -65.49},
    },
    timeout=30,
)
amendment.raise_for_status()
assert amendment.json()["updated"] == 1
amended = tap.search(
    "SELECT product_id, target_name, s_ra, s_dec FROM srcnet.data_products "
    f"WHERE project_id = '{PROJECT_ID}' AND product_id = '{amended_source_id}'"
).to_table()
amended

product_id,target_name,s_ra,s_dec
object,object,float64,float64
demo-source-000,Amended SRCNet source,62.31,-65.49


## 9. Exercise the asynchronous JSON job lifecycle

In [12]:
created_job = requests.post(
    f"{API_URL}/jobs",
    json={
        "query": (
            "SELECT product_id, s_ra, s_dec FROM srcnet.data_products "
            f"WHERE project_id = '{PROJECT_ID}' ORDER BY product_id"
        ),
        "format": "json",
        "maxrec": 100,
        "run": True,
        "run_id": "srcnet-demo-notebook",
    },
    timeout=30,
)
created_job.raise_for_status()
job = created_job.json()
job_url = f"{API_URL}/jobs/{job['job_id']}"
for _ in range(100):
    job = requests.get(job_url, timeout=10).json()
    if job["phase"] in {"COMPLETED", "ERROR", "ABORTED"}:
        break
    time.sleep(0.2)
assert job["phase"] == "COMPLETED", job
job_result = requests.get(f"{job_url}/result", timeout=30).json()
assert len(job_result["data"]) == 100
completed_jobs = requests.get(
    f"{API_URL}/jobs", params={"phase": "COMPLETED", "last": 10}, timeout=20
).json()
assert job["job_id"] in {item["job_id"] for item in completed_jobs["jobs"]}
assert requests.delete(job_url, timeout=20).status_code == 204
job

{'job_id': '6d4f096ce23938f2',
 'phase': 'COMPLETED',
 'run_id': 'srcnet-demo-notebook',
 'owner_id': None,
 'creation_time': '2026-08-22T16:34:21Z',
 'start_time': '2026-08-22T16:34:21Z',
 'end_time': '2026-08-22T16:34:22Z',
 'execution_duration': 600,
 'destruction': '2026-08-29T16:34:21Z',
 'parameters': {'LANG': 'ADQL',
  'QUERY': "SELECT product_id, s_ra, s_dec FROM srcnet.data_products WHERE project_id = 'demo-100-sources' ORDER BY product_id",
  'RUNID': 'srcnet-demo-notebook',
  'MAXREC': '100',
  'RESPONSEFORMAT': 'json'},
 'urls': {'job': 'http://localhost:8080/api/v1/jobs/6d4f096ce23938f2',
  'uws': 'http://localhost:8080/tap/async/6d4f096ce23938f2'},
 'result': {'href': 'http://localhost:8080/api/v1/jobs/6d4f096ce23938f2/result',
  'mime': 'application/json',
  'size': 4587}}

## 10. Demonstrate cascading delete

A disposable software document is inserted and deleted. Its generated artifact row is removed by the database foreign-key cascade. The main 100-source project remains available for further exploration.

In [13]:
delete_uri = "ska:delete-demo:0.0.1"
delete_payload = {
    **software_payload,
    "uri": delete_uri,
    "description": "Disposable record used to demonstrate DELETE",
    "artifacts": [
        {
            "kind": "DOCKER",
            "location": "ghcr.io/ska-telescope/delete-demo:0.0.1",
            "cpu_architecture": ["amd64"],
        }
    ],
}
created = requests.post(f"{API_URL}/software", json=delete_payload, timeout=30)
created.raise_for_status()
deleted = requests.delete(f"{API_URL}/software/{delete_uri}", timeout=30)
deleted.raise_for_status()
assert deleted.json() == {"status": "deleted", "uri": delete_uri}
assert requests.get(f"{API_URL}/software/{delete_uri}", timeout=10).status_code == 404
remaining_artifacts = tap.search(
    f"SELECT uri, location FROM srcnet.software_artifacts WHERE uri = '{delete_uri}'"
).to_table()
assert len(remaining_artifacts) == 0
print("Delete succeeded and cascaded: no disposable artifact rows remain.")

Delete succeeded and cascaded: no disposable artifact rows remain.


## Optional cleanup

The notebook deliberately leaves the 100-source project and the primary software
record in place, so the service stays worth exploring after the run. The cell
below removes them — flip `CLEANUP` to `True` to make the notebook fully
self-cleaning (and re-runnable from an empty database). It also checks that the
delete cascaded through the whole observatory hierarchy, not just the root row.


In [ ]:
CLEANUP = False  # set to True to delete the demo data on the way out

if CLEANUP:
    for path in (f"notifications/{PROJECT_ID}", f"software/{SOFTWARE_URI}"):
        requests.delete(f"{API_URL}/{path}", timeout=30).raise_for_status()
    # the root DELETE cascades through observations, scheduling and execution
    # blocks, data products and artifacts: none of them may survive it
    for table, key, value in (
        ("srcnet.projects", "project_id", PROJECT_ID),
        ("srcnet.observations", "project_id", PROJECT_ID),
        ("srcnet.scheduling_blocks", "project_id", PROJECT_ID),
        ("srcnet.execution_blocks", "project_id", PROJECT_ID),
        ("srcnet.data_products", "project_id", PROJECT_ID),
        ("srcnet.artifacts", "project_id", PROJECT_ID),
        ("srcnet.software", "uri", SOFTWARE_URI),
        ("srcnet.software_artifacts", "uri", SOFTWARE_URI),
    ):
        remaining = tap.search(f"SELECT {key} FROM {table} WHERE {key} = '{value}'").to_table()
        assert len(remaining) == 0, (table, len(remaining))
    print("Demo data removed; every cascade target is empty.")
else:
    print(f"Demo data kept: project {PROJECT_ID}, software {SOFTWARE_URI}.")